In [1]:
'''
Considerações sobre o site da vivareal

O preço de aluguel/mês está em um <p </p> de classe = 'value-item__value'
Idem para o valor de condomínio e IPTU
Para o condomínio, a tag tem um atributo data-test_id = 'condoFee'

A quantidade de metros² está num <span </span> de classe = 'amenities-item-text'
atributo data-cy = "ldp-propertyFeatures-txt" que pode ser ABL ou área de piso

A localização está num <p> com class = "location-address__text" e
data-testid="location-address" 

#--------------------------#----------------------------------#

O código no viva real do galpão está em um <p> de data-cy="ldp-propertyCodes-txt"

O valor de ABL pode estar em algum lugar do texto de descrição que VOCÊ vai retirar com regex

A atualização do anúncio está em um <span> de data-testid="listing-created-date"

Com essas informações deve ser possível fazer o scrap

'''

'\nConsiderações sobre o site da vivareal\n\nO preço de aluguel/mês está em um <p </p> de classe = \'value-item__value\'\nIdem para o valor de condomínio e IPTU\nPara o condomínio, a tag tem um atributo data-test_id = \'condoFee\'\n\nA quantidade de metros² está num <span </span> de classe = \'amenities-item-text\'\natributo data-cy = "ldp-propertyFeatures-txt" que pode ser ABL ou área de piso\n\nA localização está num <p> com class = "location-address__text" e\ndata-testid="location-address" \n\n#--------------------------#----------------------------------#\n\nO código no viva real do galpão está em um <p> de data-cy="ldp-propertyCodes-txt"\n\nO valor de ABL pode estar em algum lugar do texto de descrição que VOCÊ vai retirar com regex\n\nA atualização do anúncio está em um <span> de data-testid="listing-created-date"\n\nCom essas informações deve ser possível fazer o scrap\n\n'

In [11]:
import time
import requests
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from bs4 import BeautifulSoup
import time

In [16]:
link_inicial = 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-vila-ema-zona-leste-sao-paulo-com-garagem-300m2-aluguel-RS6000-id-2840728067/?source=ranking%2Crp'
class Scrap_Viva():
    def __init__(self, link_inicial):
        self.headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8",
            "Connection": "keep-alive",
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)
        self.r = self.session.get(link_inicial)
        self.bs = BeautifulSoup(self.r.text, 'html.parser')
        
        self.link_inicial = link_inicial
        self.links = []
        self.items = {
            'ID': [],
            'Aluguel': [],
            'Condominios': [],
            'Metros': [],
            'Local': []
        }

    def request(self, link):
        self.r = self.session.get(link)
        time.sleep(5)
        self.bs = BeautifulSoup(self.r.text, 'html.parser')
        time.sleep(5)

    def search(self):
        tags = self.bs.find_all('p', class_='value-item__value')
        if len(tags) >= 2:
            aluguel = tags[0].text
            condominio = tags[1].text
        else:
            aluguel, condominio = None, None

        metros = self.bs.find('span', class_='amenities-item-text')
        metros = metros.text if metros else None

        local = self.bs.find('p', {'class': "location-address__text",
                                   'data-testid': 'location-address'})
        local = local.text if local else None

        id_tag = self.bs.find('p', {'data-cy': 'ldp-propertyCodes-txt'})
        id_ = id_tag.text if id_tag else None

        self.items['ID'].append(id_)
        self.items['Aluguel'].append(aluguel)
        self.items['Condominios'].append(condominio)
        self.items['Metros'].append(metros)
        self.items['Local'].append(local)

    def show(self):
        for i in range(len(self.items['ID'])):
            print(f"ID: {self.items['ID'][i]}")
            print(f"Aluguel: {self.items['Aluguel'][i]}")
            print(f"Condomínio: {self.items['Condominios'][i]}")
            print(f"Metros: {self.items['Metros'][i]}")
            print(f"Local: {self.items['Local'][i]}")
            print("-" * 30)
        
        
viva = Scrap_Viva(link_inicial)
viva.search()
viva.show()


    

ID: None
Aluguel: None
Condomínio: None
Metros: None
Local: None
------------------------------


# Mesmo código com o selenium

In [91]:
import time
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from bs4 import BeautifulSoup
import time
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
import re

link_inicial = 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-vila-ema-zona-leste-sao-paulo-com-garagem-300m2-aluguel-RS6000-id-2840728067/?source=ranking%2Crp'
class Scrap_Viva():
    def __init__(self, link_inicial):
        driver_path = r"C:\Users\Guilherme.Sales\Downloads\edgedriver_win64\msedgedriver.exe"
        self.service = Service(driver_path)
        self.options = Options()
        self.driver = webdriver.Edge(service=self.service, options = self.options)
        self.driver.get(link_inicial)
        self.bs = BeautifulSoup(self.driver.page_source, 'html.parser')
        
        self.link_inicial = link_inicial
        self.links = []
        self.items = {
            'ID': [],
            'Aluguel': [],
            'Condominios': [],
            'Metros': [],
            'Local': []
        }

    def request(self, link):
        time.sleep(10)
        self.driver.execute_script("window.scrollTo(1, document.body.scrollHeight);") #scrollar para baixo para carregar os imóveis
        total_height = self.driver.execute_script("return document.body.scrollHeight")
        half_height = total_height / 2
        self.driver.execute_script(f"window.scrollTo(0, {half_height});")
        self.driver.quit()
        self.service = Service(driver_path)
        self.options = Options()
        self.driver = webdriver.Edge(service=self.service, options=self.options)  # Reabre com novo User-Agent
        self.driver.get(link)
        WebDriverWait(self.driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "value-item__value"))
)
        self.bs = BeautifulSoup(self.driver.page_source, 'html.parser')
        time.sleep(5)

    def search(self):
        tags = self.bs.find_all('p', class_='value-item__value')
        if len(tags) >= 2:
            aluguel = tags[0].text
            condominio = tags[1].text
        else:
            aluguel, condominio = None, None

        metros = self.bs.find('span', class_='amenities-item-text')
        metros = metros.text if metros else None

        local = self.bs.find('p', {'class': "location-address__text",
                                   'data-testid': 'location-address'})
        local = local.text if local else None

        id_tag = self.bs.find('p', {'data-cy': 'ldp-propertyCodes-txt'})
        id_ = id_tag.text if id_tag else None

        self.items['ID'].append(id_)
        self.items['Aluguel'].append(aluguel)
        self.items['Condominios'].append(condominio)
        self.items['Metros'].append(metros)
        self.items['Local'].append(local)

    def show(self):
        for i in range(len(self.items['ID'])):
            print(f"ID: {self.items['ID'][i]}")
            print(f"Aluguel: {self.items['Aluguel'][i]}")
            print(f"Condomínio: {self.items['Condominios'][i]}")
            print(f"Metros: {self.items['Metros'][i]}")
            print(f"Local: {self.items['Local'][i]}")
            print("-" * 30)

    def get_links(self):
        self.driver.execute_script("window.scrollTo(8, document.body.scrollHeight);") #scrollar para baixo para carregar os imóveis
        wait = WebDriverWait(self.driver, 20)
        elem = wait.until(
        EC.presence_of_element_located(
        (By.CSS_SELECTOR, ".olx-core-carousel__viewport.h-full")
    )
)
        # pega todos os links dentro dessa seção
        elements = self.driver.find_elements(By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel']")
        self.links = [el.get_attribute("href") for el in elements]

        print(self.links)

    def segue_links(self):
        for link in self.links:
            #self.driver.execute_script("window.scrollTo((25, 300), document.body.scrollHeight);") #scrollar para baixo para carregar os imóveis
            time.sleep(5)
            self.request(link)
            self.search()
            self.show()

                
viva = Scrap_Viva(link_inicial)
viva.search()
viva.show()
viva.get_links()
viva.segue_links()

ID: (Código do anunciante: BC30505 | Código no Viva Real: 2840728067)
Aluguel: R$ 6.000/mês
Condomínio: Isento
Metros: 300 m² 
Local: Rua Málaga, 208 - Vila Ema, São Paulo - SP
------------------------------
['https://www.vivareal.com.br/imovel/sobrado-4-quartos-alto-da-mooca-zona-leste-sao-paulo-com-garagem-360m2-aluguel-RS8000-id-2529952177/?source=showcase%2Cldp', 'https://www.vivareal.com.br/imovel/sobrado-3-quartos-nova-gerti-bairros-sao-caetano-do-sul-com-garagem-300m2-aluguel-RS6000-id-2841204396/?source=showcase%2Cldp', 'https://www.vivareal.com.br/imovel/sobrado-3-quartos-vila-das-merces-zona-sul-sao-paulo-com-garagem-250m2-aluguel-RS4000-id-2558589486/?source=showcase%2Cldp', 'https://www.vivareal.com.br/imovel/casa-de-condominio-3-quartos-cambuci-zona-sul-sao-paulo-com-garagem-300m2-aluguel-RS8000-id-2841101242/?source=showcase%2Cldp', 'https://www.vivareal.com.br/imovel/flat-2-quartos-paraiso-zona-sul-sao-paulo-com-garagem-60m2-aluguel-RS4644-id-2653241445/?source=showcase%

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: MicrosoftEdge=140.0.3485.94); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	GetHandleVerifier [0x0x7ff795b23ec5+18309]
	(No symbol) [0x0x7ff795a91570]
	GetHandleVerifier [0x0x7ff795d18efc+2070460]
	(No symbol) [0x0x7ff795880ad0]
	(No symbol) [0x0x7ff79589f85a]
	(No symbol) [0x0x7ff7959053f4]
	(No symbol) [0x0x7ff79591c9ba]
	(No symbol) [0x0x7ff7958ff943]
	(No symbol) [0x0x7ff7958d38e6]
	(No symbol) [0x0x7ff7958d2b52]
	(No symbol) [0x0x7ff7958d3723]
	(No symbol) [0x0x7ff7959c2f75]
	(No symbol) [0x0x7ff7959ce74d]
	GetHandleVerifier [0x0x7ff795bb95b3+630387]
	GetHandleVerifier [0x0x7ff795bc1151+662033]
	(No symbol) [0x0x7ff795a9d659]
	(No symbol) [0x0x7ff795a96ff4]
	(No symbol) [0x0x7ff795a97143]
	(No symbo

# Versão Atual de 06/10


In [38]:
import time
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from bs4 import BeautifulSoup
import time
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
import pandas as pd
import openpyxl

link_exemplo = 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-vila-ema-zona-leste-sao-paulo-com-garagem-300m2-aluguel-RS6000-id-2840728067/?source=ranking%2Crp'
link_inicial = 'https://www.vivareal.com.br/aluguel/sp/guarulhos/galpao_comercial/?transacao=aluguel&onde=%2CS%C3%A3o+Paulo%2CGuarulhos%2C%2C%2C%2C%2Ccity%2CBR%3ESao+Paulo%3ENULL%3EGuarulhos%2C-23.454314%2C-46.533664%2C&tipos=galpao_comercial&areaMinima=500'

class Scrap_Viva():
    def __init__(self, link_inicial):
        self.driver_path = r"C:\Users\Guilherme.Sales\Downloads\edgedriver_win64\msedgedriver.exe"
        self.service = Service(self.driver_path)
        self.options = Options()
        self.options.add_experimental_option("excludeSwitches", ["enable-logging"]) #Chato pra caralho
        self.driver = webdriver.Edge(service=self.service, options = self.options)
        self.driver.get(link_inicial)
        self.bs = BeautifulSoup(self.driver.page_source, 'html.parser')
        
        self.link_inicial = link_inicial
        self.link_externo = link_inicial
        self.links = []
        self.df = pd.DataFrame(columns=['ID', 'Aluguel', 'Condominios', 'Metros', 'Local', 'Link'])

    def request(self, link):
        self.driver_path = r"C:\Users\Guilherme.Sales\Downloads\edgedriver_win64\msedgedriver.exe"
        total_height = self.driver.execute_script("return document.body.scrollHeight")
        half_height = total_height / 2
        self.driver.execute_script(f"window.scrollTo(0, {half_height});")
        self.service = Service(self.driver_path)
        self.options = Options()
        self.driver = webdriver.Edge(service=self.service, options=self.options)  # Reabre com novo User-Agent
        self.driver.get(link)
        self.bs = BeautifulSoup(self.driver.page_source, 'html.parser')
        time.sleep(5)

    def search(self, link):
        html = self.driver.page_source
        bs = BeautifulSoup(html, 'html.parser')

        tags = bs.find_all('p', class_='value-item__value')
        aluguel = tags[0].get_text(strip=True) if len(tags) >= 1 else None
        condominio = tags[1].get_text(strip=True) if len(tags) >= 2 else None

        metros_tag = bs.find('span', class_='amenities-item-text')
        metros = metros_tag.get_text(strip=True) if metros_tag else None

        local_tag = bs.find('p', {'class': "location-address__text", 'data-testid': 'location-address'})
        local = local_tag.get_text(strip=True) if local_tag else None

        id_tag = bs.find('p', {'data-cy': 'ldp-propertyCodes-txt'})
        id_ = id_tag.get_text(strip=True) if id_tag else None

        att_tag = bs.find()

        link_atual = self.driver.current_url

        dados = {
            'ID': id_,
            'Aluguel': aluguel,
            'Condominios': condominio,
            'Metros': metros,
            'Local': local,
            'Link': link_atual
        }
        self.df = pd.concat([self.df, pd.DataFrame([dados])], ignore_index=True)
        # viva.df.to_excel("imoveis_vivareal.xlsx", index=False)
        
            try:
                self.driver.quit()
            except Exception:
                pass

    def show(self):
        print(self.df.tail(1))

    def get_links(self):
        WebDriverWait(self.driver, 10).until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/galpao']")
    )
)        
        # pega todos os links dentro dessa seção
        elements = self.driver.find_elements(By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/galpao']")
        self.links = [el.get_attribute("href") for el in elements]

        print(self.links)

    def segue_links(self):
        for page_num in range(1, 50):
            self.get_links()
            time.sleep(5)
            for link in self.links:
                time.sleep(5)
                self.request(link)
                WebDriverWait(self.driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "value-item__value"))
)
                self.search(link)
                self.show()
            self.link_externo = self.link_inicial + '&pagina={}'.format(page_num)
            self.request(self.link_externo)
                
viva = Scrap_Viva(link_inicial)
viva.segue_links()

['https://www.vivareal.com.br/imovel/galpao-deposito-armazem-cidade-industrial-satelite-de-sao-paulo-guarulhos-com-garagem-7976m2-venda-RS45000000-id-2839108229/?source=ranking%2Crp', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-jardim-aida-guarulhos-510m2-aluguel-RS16000-id-2840073725/?source=ranking%2Crp', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-residencial-parque-cumbica-guarulhos-com-garagem-34174m2-aluguel-RS1366960-id-2768220301/?source=ranking%2Crp', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-jardim-giovana-guarulhos-com-garagem-1200m2-aluguel-RS43000-id-2829805435/?source=ranking%2Crp', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-vila-endres-guarulhos-500m2-aluguel-RS15000-id-2837974115/?source=ranking%2Crp', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-varzea-do-palacio-guarulhos-com-garagem-1380m2-aluguel-RS42700-id-2838399514/?source=ranking%2Crp', 'https://www.vivareal.com.br/imovel/galpao-de

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: MicrosoftEdge=141.0.3537.57); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	GetHandleVerifier [0x0x7ff60d943ec5+18309]
	(No symbol) [0x0x7ff60d8b1570]
	GetHandleVerifier [0x0x7ff60db38efc+2070460]
	(No symbol) [0x0x7ff60d6a0ad0]
	(No symbol) [0x0x7ff60d6bf85a]
	(No symbol) [0x0x7ff60d7253f4]
	(No symbol) [0x0x7ff60d73c9ba]
	(No symbol) [0x0x7ff60d71f943]
	(No symbol) [0x0x7ff60d6f38e6]
	(No symbol) [0x0x7ff60d6f2b52]
	(No symbol) [0x0x7ff60d6f3723]
	(No symbol) [0x0x7ff60d7e2f75]
	(No symbol) [0x0x7ff60d7ee74d]
	GetHandleVerifier [0x0x7ff60d9d95b3+630387]
	GetHandleVerifier [0x0x7ff60d9e1151+662033]
	(No symbol) [0x0x7ff60d8bd659]
	(No symbol) [0x0x7ff60d8b6ff4]
	(No symbol) [0x0x7ff60d8b7143]
	(No symbo

# Códigos abaixo apenas de referência, odeio chatgpt

In [ ]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

class Scrap_Viva:
    def __init__(self, link_inicial):
        self.driver_path = r"C:\Users\Guilherme.Sales\Downloads\edgedriver_win64\msedgedriver.exe"
        self.service = Service(self.driver_path)
        self.options = Options()
        self.options.add_experimental_option("excludeSwitches", ["enable-logging"])
        # driver fixo para a listagem
        self.list_driver = webdriver.Edge(service=self.service, options=self.options)
        self.wait_list = WebDriverWait(self.list_driver, 15)

        self.link_inicial = link_inicial
        self.df = pd.DataFrame(columns=['ID', 'Aluguel', 'Condominios', 'Metros', 'Local', 'Link'])

    def open_list_page(self, url):
        self.list_driver.get(url)
        # aguarda a grade de imóveis carregar
        self.wait_list.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/galpao']")
            )
        )

    def get_links_on_page(self):
        elements = self.list_driver.find_elements(By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/galpao']")
        # evita duplicados e URLs de tracking com parâmetros diferentes
        raw = [el.get_attribute("href") for el in elements]
        unique = list(dict.fromkeys(raw))
        return unique

    def search(self, link):
        """Abre um driver efêmero, extrai dados e fecha SEMPRE."""
        detail_driver = None
        try:
            detail_driver = webdriver.Edge(service=Service(self.driver_path), options=self.options)
            wait_detail = WebDriverWait(detail_driver, 15)
            detail_driver.get(link)

            # rolar um pouco para disparar render dinâmico (se necessário)
            detail_driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.5);")

            # aguarda valores principais
            wait_detail.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "value-item__value")))
            html = detail_driver.page_source
            bs = BeautifulSoup(html, 'html.parser')

            tags = bs.find_all('p', class_='value-item__value')
            aluguel = tags[0].get_text(strip=True) if len(tags) >= 1 else None
            condominio = tags[1].get_text(strip=True) if len(tags) >= 2 else None

            metros_tag = bs.find('span', class_='amenities-item-text')
            metros = metros_tag.get_text(strip=True) if metros_tag else None

            local_tag = bs.find('p', {'class': "location-address__text", 'data-testid': 'location-address'})
            local = local_tag.get_text(strip=True) if local_tag else None

            id_tag = bs.find('p', {'data-cy': 'ldp-propertyCodes-txt'})
            id_ = id_tag.get_text(strip=True) if id_tag else None

            att_tag = bs.find()

            link_atual = detail_driver.current_url

            dados = {
                'ID': id_,
                'Aluguel': aluguel,
                'Condominios': condominio,
                'Metros': metros,
                'Local': local,
                'Link': link_atual
            }
            self.df = pd.concat([self.df, pd.DataFrame([dados])], ignore_index=True)
            # viva.df.to_excel("imoveis_vivareal.xlsx", index=False)

        finally:
            # garante que o driver do detalhe FECHA MESMO em caso de erro
            if detail_driver is not None:
                try:
                    detail_driver.quit()
                except Exception:
                    pass

    def segue_links(self, num_paginas=50, pausa_entre_links=3.0):
        # abre a primeira página da listagem
        self.open_list_page(self.link_inicial)

        for page_num in range(1, num_paginas + 1):
            # coleta links desta página
            links = self.get_links_on_page()

            for link in links:
                self.search(link)
                print(self.df.tail(1))
                print(link)
                time.sleep(pausa_entre_links)

            # próxima página da listagem
            prox_url = f"{self.link_inicial}&pagina={page_num+1}"
            self.open_list_page(prox_url)

    def close(self):
        try:
            self.list_driver.quit()
        except Exception:
            pass

link_inicial = "https://www.vivareal.com.br/aluguel/sp/guarulhos/galpao_comercial/?transacao=aluguel&onde=%2CS%C3%A3o+Paulo%2CGuarulhos%2C%2C%2C%2C%2Ccity%2CBR%3ESao+Paulo%3ENULL%3EGuarulhos"
link_inicial = 'https://www.vivareal.com.br/aluguel/sp/sao-paulo/galpao_comercial/?transacao=aluguel&onde=%2CS%C3%A3o+Paulo%2CS%C3%A3o+Paulo%2C%2C%2C%2C%2Ccity%2CBR%3ESao+Paulo%3ENULL%3ESao+Paulo%2C-23.555771%2C-46.639557%2C&tipos=galpao_comercial&areaMinima=500'
viva = Scrap_Viva(link_inicial)
try:
    viva.segue_links(num_paginas=15)  
finally:
    viva.close()

# viva.df.to_excel("imoveis_vivareal.xlsx", index=False)


In [37]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

class Scrap_Viva:
    def __init__(self, link_inicial):
        self.driver_path = r"C:\Users\Guilherme.Sales\Downloads\edgedriver_win64\msedgedriver.exe"
        self.service = Service(self.driver_path)
        self.options = Options()
        self.options.add_experimental_option("excludeSwitches", ["enable-logging"])
        # driver fixo para a listagem
        self.list_driver = webdriver.Edge(service=self.service, options=self.options)
        self.wait_list = WebDriverWait(self.list_driver, 15)

        self.link_inicial = link_inicial
        self.df = pd.DataFrame(columns=['ID', 'Aluguel', 'Condominios', 'Metros', 'Local', 'Link'])

    def open_list_page(self, url):
        self.list_driver.get(url)
        # aguarda a grade de imóveis carregar
        self.wait_list.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/galpao']")
            )
        )
        time.sleep(random.uniform(3, 10))

    def get_links_on_page(self, max_retries=5, pause=1.0):
        links = []
        for i in range(max_retries):
            self.list_driver.execute_script("window.scrollTo(0, document.body.scrollHeight * arguments[0] / 10);", i+4)
            time.sleep(pause)
            elems = self.list_driver.find_elements(By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/']")
            hrefs = [e.get_attribute("href") for e in elems if e.get_attribute("href")]
            # mantém só galpão + presença de '-id-'
            hrefs = [h.split('?')[0] for h in hrefs if "/imovel/galpao" in h and "-id-" in h and "vivareal.com.br" in h]
            if hrefs:
                links = list(dict.fromkeys(hrefs))
                break
        return links


    def search(self, link):
        """Abre um driver efêmero, extrai dados e fecha SEMPRE."""
        detail_driver = None
        try:
            detail_driver = webdriver.Edge(service=Service(self.driver_path), options=self.options)
            wait_detail = WebDriverWait(detail_driver, 15)
            detail_driver.get(link)

            # rolar um pouco para disparar render dinâmico (se necessário)
            detail_driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.5);")

            # aguarda valores principais
            wait_detail.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "value-item__value")))
            html = detail_driver.page_source
            bs = BeautifulSoup(html, 'html.parser')

            tags = bs.find_all('p', class_='value-item__value')
            aluguel = tags[0].get_text(strip=True) if len(tags) >= 1 else None
            condominio = tags[1].get_text(strip=True) if len(tags) >= 2 else None

            metros_tag = bs.find('span', class_='amenities-item-text')
            metros = metros_tag.get_text(strip=True) if metros_tag else None

            local_tag = bs.find('p', {'class': "location-address__text", 'data-testid': 'location-address'})
            local = local_tag.get_text(strip=True) if local_tag else None

            id_tag = bs.find('p', {'data-cy': 'ldp-propertyCodes-txt'})
            id_ = id_tag.get_text(strip=True) if id_tag else None

            att_tag = bs.find()

            link_atual = detail_driver.current_url

            dados = {
                'ID': id_,
                'Aluguel': aluguel,
                'Condominios': condominio,
                'Metros': metros,
                'Local': local,
                'Link': link_atual
            }
            self.df = pd.concat([self.df, pd.DataFrame([dados])], ignore_index=True)
            # viva.df.to_excel("imoveis_vivareal.xlsx", index=False)

        finally:
            # garante que o driver do detalhe FECHA MESMO em caso de erro
            if detail_driver is not None:
                try:
                    detail_driver.quit()
                except Exception:
                    pass

    def segue_links(self, num_paginas=50, pausa_entre_links=3.0):
        # abre a primeira página da listagem
        self.open_list_page(self.link_inicial)

        for page_num in range(1, num_paginas + 1):
            # coleta links desta página
            print('Número da página: {}'.format(page_num))
            links = self.get_links_on_page()
            print(links, 'tamanho da lista de links: {}'.format(len(links)))

            for link in links:
                self.search(link)
                print(self.df.tail(1))
                print(link)
                time.sleep(pausa_entre_links)

            # próxima página da listagem
            prox_url = f"{self.link_inicial}&pagina={page_num+1}"
            self.open_list_page(prox_url)

    def close(self):
        try:
            self.list_driver.quit()
        except Exception:
            pass



link_inicial = "https://www.vivareal.com.br/aluguel/sp/guarulhos/galpao_comercial/?transacao=aluguel&onde=%2CS%C3%A3o+Paulo%2CGuarulhos%2C%2C%2C%2C%2Ccity%2CBR%3ESao+Paulo%3ENULL%3EGuarulhos"
link_inicial = 'https://www.vivareal.com.br/aluguel/sp/sao-paulo/galpao_comercial/?transacao=aluguel&onde=%2CS%C3%A3o+Paulo%2CS%C3%A3o+Paulo%2C%2C%2C%2C%2Ccity%2CBR%3ESao+Paulo%3ENULL%3ESao+Paulo%2C-23.555771%2C-46.639557%2C&tipos=galpao_comercial&areaMinima=500'
viva = Scrap_Viva(link_inicial)
try:
    viva.segue_links(num_paginas=25)  
finally:
    viva.close()

# viva.df.to_excel("imoveis_vivareal.xlsx", index=False)

Número da página: 1
[] tamanho da lista de links: 0
Número da página: 2
[] tamanho da lista de links: 0
Número da página: 3
[] tamanho da lista de links: 0
Número da página: 4


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: MicrosoftEdge=141.0.3537.57); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	GetHandleVerifier [0x0x7ff60d943ec5+18309]
	(No symbol) [0x0x7ff60d8b1570]
	GetHandleVerifier [0x0x7ff60db38efc+2070460]
	(No symbol) [0x0x7ff60d6a0ad0]
	(No symbol) [0x0x7ff60d6bf85a]
	(No symbol) [0x0x7ff60d7253f4]
	(No symbol) [0x0x7ff60d73c9ba]
	(No symbol) [0x0x7ff60d71f943]
	(No symbol) [0x0x7ff60d6f38e6]
	(No symbol) [0x0x7ff60d6f2b52]
	(No symbol) [0x0x7ff60d6f3723]
	(No symbol) [0x0x7ff60d7e2f75]
	(No symbol) [0x0x7ff60d7ee74d]
	GetHandleVerifier [0x0x7ff60d9d95b3+630387]
	GetHandleVerifier [0x0x7ff60d9e1151+662033]
	(No symbol) [0x0x7ff60d8bd659]
	(No symbol) [0x0x7ff60d8b6ff4]
	(No symbol) [0x0x7ff60d8b7143]
	(No symbo

In [26]:
import random

In [29]:
random.uniform(3, 10)

3.72655802464979

# Versão com Click de Mouse

In [32]:
import time, random
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
)

def wait_dom_ready(driver, timeout=12):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )

def wait_page_changed(driver, old_url, old_first_href, timeout=12):
    w = WebDriverWait(driver, timeout)
    # 1) tenta URL diferente
    try:
        w.until(lambda d: d.current_url != old_url)
        return True
    except TimeoutException:
        pass
    # 2) tenta mudança do primeiro card
    try:
        new_first = driver.find_elements(By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/']")
        if not new_first:
            return False
        curr = new_first[0].get_attribute("href")
        return curr != old_first_href
    except StaleElementReferenceException:
        return True

class Scrap_Viva:
    def __init__(self, link_inicial):
        self.driver_path = r"C:\Users\Guilherme.Sales\Downloads\edgedriver_win64\msedgedriver.exe"
        self.service = Service(self.driver_path)
        self.options = Options()
        self.options.add_experimental_option("excludeSwitches", ["enable-logging"])
        self.list_driver = webdriver.Edge(service=self.service, options=self.options)
        self.wait_list = WebDriverWait(self.list_driver, 15)
        self.link_inicial = link_inicial
        self.df = pd.DataFrame(columns=['ID', 'Aluguel', 'Condominios', 'Metros', 'Local', 'Link'])

    def open_list_page(self, url):
        self.list_driver.get(url)
        wait_dom_ready(self.list_driver)
        # aguarda haver algum card de imóvel (seletor relaxado)
        self.wait_list.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/']")
            )
        )
        time.sleep(random.uniform(1.0, 2.2))  # micro-jitter

    def get_links_on_page(self, max_retries=5, pause=0.8):
        links = []
        for i in range(max_retries):
            self.list_driver.execute_script(
                "window.scrollTo(0, document.body.scrollHeight * arguments[0] / 10);", i+4
            )
            time.sleep(pause)
            elems = self.list_driver.find_elements(By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/']")
            hrefs = [e.get_attribute("href") for e in elems if e.get_attribute("href")]
            hrefs = [h.split('?')[0] for h in hrefs if "/imovel/galpao" in h and "-id-" in h]
            if hrefs:
                links = list(dict.fromkeys(hrefs))
                break
        return links

    def search(self, link):
        detail_driver = None
        try:
            detail_driver = webdriver.Edge(service=Service(self.driver_path), options=self.options)
            wait_detail = WebDriverWait(detail_driver, 15)
            detail_driver.get(link)
            detail_driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.5);")
            wait_detail.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "value-item__value")))
            bs = BeautifulSoup(detail_driver.page_source, 'html.parser')

            tags = bs.find_all('p', class_='value-item__value')
            aluguel = tags[0].get_text(strip=True) if len(tags) >= 1 else None
            condominio = tags[1].get_text(strip=True) if len(tags) >= 2 else None
            metros_tag = bs.find('span', class_='amenities-item-text')
            metros = metros_tag.get_text(strip=True) if metros_tag else None
            local_tag = bs.find('p', {'class': "location-address__text", 'data-testid': 'location-address'})
            local = local_tag.get_text(strip=True) if local_tag else None
            id_tag = bs.find('p', {'data-cy': 'ldp-propertyCodes-txt'})
            id_ = id_tag.get_text(strip=True) if id_tag else None

            dados = {
                'ID': id_,
                'Aluguel': aluguel,
                'Condominios': condominio,
                'Metros': metros,
                'Local': local,
                'Link': detail_driver.current_url
            }
            self.df = pd.concat([self.df, pd.DataFrame([dados])], ignore_index=True)
        finally:
            if detail_driver is not None:
                try: detail_driver.quit()
                except Exception: pass

    def click_proxima_pagina(self):
        sels = [
            "a[rel='next']",
            "a[aria-label*='Próxima']",
            "li.pagination__item--next a",
            "a[data-testid='pagination-forward']",
            "a[title*='Próxima']",
        ]
        for sel in sels:
            btns = self.list_driver.find_elements(By.CSS_SELECTOR, sel)
            if not btns:
                continue
            btn = btns[0]
            self.list_driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            try:
                old_url = self.list_driver.current_url
                first_elems = self.list_driver.find_elements(By.CSS_SELECTOR, "a[href^='https://www.vivareal.com.br/imovel/']")
                old_first = first_elems[0].get_attribute("href") if first_elems else None
                btn.click()
                if wait_page_changed(self.list_driver, old_url, old_first, timeout=12):
                    return True
            except (ElementClickInterceptedException, StaleElementReferenceException):
                time.sleep(0.8)
                continue
        return False

    def segue_links(self, num_paginas=50, pausa_entre_links=3.0):
        # abre a primeira página
        self.open_list_page(self.link_inicial)

        for page_num in range(1, num_paginas + 1):
            print(f'Número da página: {page_num}')
            links = self.get_links_on_page()
            print(links, f'tamanho da lista de links: {len(links)}')

            if not links:
                print("[INFO] Sem links nesta página. Encerrando.")
                break

            for link in links:
                self.search(link)
                print(self.df.tail(1))
                print(link)
                time.sleep(pausa_entre_links)

            # tenta próxima; se não houver, encerra
            moved = self.click_proxima_pagina()
            if not moved:
                print("[INFO] Não avançou para a próxima página. Fim.")
                break

    def close(self):
        try:
            self.list_driver.quit()
        except Exception:
            pass

viva = Scrap_Viva(link_inicial)
try:
    viva.segue_links(num_paginas=25)  
finally:
    viva.close()


Número da página: 1
['https://www.vivareal.com.br/imovel/galpao-deposito-armazem-santa-efigenia-sao-paulo-com-garagem-1714m2-venda-RS8400000-id-2805714427/', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-casa-verde-sao-paulo-com-garagem-909m2-venda-RS3300000-id-2828301357/', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-parque-industrial-tomas-edson-sao-paulo-com-garagem-2150m2-aluguel-RS95000-id-2829213635/', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-lapa-sao-paulo-800m2-aluguel-RS58900-id-2841511741/', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-caxingui-sao-paulo-com-garagem-1268m2-aluguel-RS55000-id-2743720133/', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-barra-funda-sao-paulo-com-garagem-3300m2-venda-RS33000000-id-2811594815/', 'https://www.vivareal.com.br/imovel/galpao-deposito-armazem-lapa-de-baixo-sao-paulo-com-garagem-4571m2-aluguel-RS182840-id-2800809433/', 'https://www.vivareal.com.br/imovel/galpa

In [33]:
link_inicial

'https://www.vivareal.com.br/aluguel/sp/sao-paulo/galpao_comercial/?transacao=aluguel&onde=%2CS%C3%A3o+Paulo%2CS%C3%A3o+Paulo%2C%2C%2C%2C%2Ccity%2CBR%3ESao+Paulo%3ENULL%3ESao+Paulo%2C-23.555771%2C-46.639557%2C&tipos=galpao_comercial&areaMinima=500'